kernel : Python (Pixi)

In [ ]:
%load_ext autoreload
%autoreload 2
import gc
import numpy as np
import os
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

raw_data_dir = find_env_dir("RAW_DATA")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

SERIES = "velmeshev"

adata = sc.read_h5ad(os.path.join(raw_data_dir, SERIES, "2001fbc7-de84-4e84-9a76-4290bcd65415.h5ad"))
adata.obs

,tissue_ontology_term_id,assay_ontology_term_id,disease_ontology_term_id,cell_type_ontology_term_id,self_reported_ethnicity_ontology_term_id,development_stage_ontology_term_id,sex_ontology_term_id,donor_id,suspension_type,is_primary_data,...,PMI,tissue_type,cell_type,assay,disease,sex,tissue,self_reported_ethnicity,development_stage,observation_joinid
U01_AAACCTGAGAAACCAT-1_5387_BA9,UBERON:0001870,EFO:0009899,PATO:0000461,CL:0002453,unknown,HsapDv:0000106,PATO:0000384,5387,nucleus,True,...,13,tissue,oligodendrocyte precursor cell,10x 3' v2,normal,male,frontal cortex,unknown,12-year-old stage,NLsy@+9O(!
U01_AAACCTGAGACTCGGA-1_5387_BA9,UBERON:0001870,EFO:0009899,PATO:0000461,unknown,unknown,HsapDv:0000106,PATO:0000384,5387,nucleus,True,...,13,tissue,unknown,10x 3' v2,normal,male,frontal cortex,unknown,12-year-old stage,?z*%FX07mf
U01_AAACCTGCAGGGATTG-1_5387_BA9,UBERON:0001870,EFO:0009899,PATO:0000461,CL:0002319,unknown,HsapDv:0000106,PATO:0000384,5387,nucleus,True,...,13,tissue,neural cell,10x 3' v2,normal,male,frontal cortex,unknown,12-year-old stage,`xdGX($pq=
U01_AAACCTGCAGGTCGTC-1_5387_BA9,UBERON:0001870,EFO:0009899,PATO:0000461,CL:0002453,unknown,HsapDv:0000106,PATO:0000384,5387,nucleus,True,...,13,tissue,oligodendrocyte precursor cell,10x 3' v2,normal,male,frontal cortex,unknown,12-year-old stage,gNR8C$@9v7
U01_AAACCTGGTACGCTGC-1_5387_BA9,UBERON:0001870,EFO:0009899,PATO:0000461,CL:0002319,unknown,HsapDv:0000106,PATO:0000384,5387,nucleus,True,...,13,tissue,neural cell,10x 3' v2,normal,male,frontal cortex,unknown,12-year-old stage,$gA}zc;hB|
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Trevino_hft_ctx_w21_dc2r2_r2_TTTGGTGCATGAATAG,UBERON:0000956,EFO:0030059,PATO:0000461,CL:0002319,unknown,HsapDv:0000199,PATO:0000383,hft_ctx_w21,nucleus,True,...,0,tissue,neural cell,10x multiome,normal,female,cerebral cortex,unknown,fifth LMP month stage,<ORD$w3Yh0
Trevino_hft_ctx_w21_dc2r2_r2_TTTGTCCCAGTTTACG,UBERON:0000956,EFO:0030059,PATO:0000461,CL:0002319,unknown,HsapDv:0000199,PATO:0000383,hft_ctx_w21,nucleus,True,...,0,tissue,neural cell,10x multiome,normal,female,cerebral cortex,unknown,fifth LMP month stage,_3nHk=UzT1
Trevino_hft_ctx_w21_dc2r2_r2_TTTGTCTAGTGAGCAA,UBERON:0000956,EFO:0030059,PATO:0000461,CL:0002319,unknown,HsapDv:0000199,PATO:0000383,hft_ctx_w21,nucleus,True,...,0,tissue,neural cell,10x multiome,normal,female,cerebral cortex,unknown,fifth LMP month stage,$4J05Hbf&$
Trevino_hft_ctx_w21_dc2r2_r2_TTTGTGAAGACAGGCG,UBERON:0000956,EFO:0030059,PATO:0000461,CL:0002319,unknown,HsapDv:0000199,PATO:0000383,hft_ctx_w21,nucleus,True,...,0,tissue,neural cell,10x multiome,normal,female,cerebral cortex,unknown,fifth LMP month stage,2E5W_s1F>&


In [2]:
assert isinstance(adata.obs, pd.DataFrame)
adata.obs["tissue"] = adata.obs["tissue"]
adata.obs["assay"] = adata.obs["assay"]
adata.obs["celltype_ref"] = adata.obs["cell_type"]
adata.obs["disease"] = adata.obs["disease"]
adata.obs["pmi"] = adata.obs["PMI"]
adata.obs["sex"] = adata.obs["sex"]
adata.obs["development_stage"] = adata.obs["development_stage"]
adata.obs["donor"] = adata.obs["donor_id"]
adata.obs["sample"] = adata.obs["sample"]
adata.obs["region"] = adata.obs["region"]
adata.obs["series"] = adata.obs["dataset"]

keep = ["tissue", "assay", "celltype_ref", "disease", "pmi", "sex", "development_stage", "donor", "sample", "region", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [ ]:
adata.var["ensembl_id"] = adata.var_names
adata.var_names = pd.Index(adata.var["feature_name"].astype(str)) #type: ignore
adata.var_names_make_unique()

In [5]:
adata.X = csr_matrix(adata.raw.X)
adata.raw = None #type: ignore

In [8]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.var.index.name = None
adata.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_raw.h5ad"))